In [1]:
"""BigQuery-based version of `analyse_linearity_gcs.ipynb`.

This notebook:
- Loads object labels (`id` → `object_id`) from a local analysis_observations parquet file.
- Scans transformed detections stored in a BigQuery table, grouped by test orbit.
- For each (test orbit, object_id) with ≥ MIN_OBS_PER_ORBIT detections, computes a simple
  linearity measure in (theta_x, theta_y) space.
- Periodically checkpoints results to a local parquet file.

NOTE: You will need to adjust DATASET_ID, TABLE_ID, and possibly column names
once the final BigQuery schema is confirmed.
"""

import os
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

from IPython.display import display

# Configuration

# BigQuery configuration
PROJECT_ID = "moeyens-thor-dev"

# Analysis observations table (source_catalog_nov25 has id and object_id)
ANALYSIS_OBS_DATASET = "thor_v3"
ANALYSIS_OBS_TABLE = "source_catalog_nov25"
FULL_ANALYSIS_OBS_TABLE_ID = f"{PROJECT_ID}.{ANALYSIS_OBS_DATASET}.{ANALYSIS_OBS_TABLE}"

# Transformed detections table (thor_dec3_transformed_detections)
TRANSFORMED_DATASET = "thor_v3"
TRANSFORMED_TABLE = "thor_dec3_transformed_detections"
FULL_TRANSFORMED_TABLE_ID = f"{PROJECT_ID}.{TRANSFORMED_DATASET}.{TRANSFORMED_TABLE}"

# Linearity / filtering configuration
MIN_OBS_PER_ORBIT = 6
CHECKPOINT_PATH = "/Users/b612foundation/b612_packages/data/linearity_results_bq.parquet"
CHECKPOINT_EVERY = 10  # checkpoint every N test orbits



In [ ]:
# Linearity computation helper

def compute_linearity(group: pd.DataFrame) -> pd.Series:
    """Compute a simple linearity metric in (theta_x, theta_y) for one object.

    Assumes a column "coordinates" that is a nested dict/struct with
    fields including "theta_x" and "theta_y".
    """
    # unpack nested coordinates into plain columns
    coords = pd.json_normalize(group["coordinates"])
    x = coords["theta_x"].to_numpy()
    y = coords["theta_y"].to_numpy()
    n = len(x)
    if n < 2:
        return pd.Series({"n_obs": n, "linearity_r2": np.nan})

    # least-squares line fit: y ≈ m x + b
    m, b = np.polyfit(x, y, 1)
    y_hat = m * x + b

    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0

    return pd.Series({"n_obs": n, "linearity_r2": r2})



In [3]:
# BigQuery client and list of test orbits

bqclient = bigquery.Client(project=PROJECT_ID)

print(f"Listing distinct test_orbit_id values from {FULL_TRANSFORMED_TABLE_ID}...")

# NOTE: assumes a column "test_orbit_id" exists in the table.
# Adjust if the actual column name is different.
test_orbits_sql = f"""
SELECT DISTINCT test_orbit_id
FROM `{FULL_TRANSFORMED_TABLE_ID}`
ORDER BY test_orbit_id
"""

test_orbit_ids = [row.test_orbit_id for row in bqclient.query(test_orbits_sql)]
print(f"Found {len(test_orbit_ids)} candidate test orbits.")



/Users/b612foundation/b612_packages/adam-thor-research/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Listing distinct test_orbit_id values from moeyens-thor-dev.thor_v3.thor_dec3_transformed_detections...
Found 5145 candidate test orbits.


In [4]:
# Main loop: per-test-orbit analysis

all_summaries = []

for i, test_orbit_id in enumerate(test_orbit_ids, start=1):
    print(f"Processing {i} of {len(test_orbit_ids)} (test_orbit_id={test_orbit_id})")

    # Join transformed_detections with analysis_observations in BigQuery,
    # filter out noise (WHERE object_id IS NOT NULL), and only pull labeled detections.
    query = f"""
    WITH labeled_detections AS (
      SELECT
        td.id,
        td.night,
        td.coordinates,
        td.state_id,
        td.test_orbit_id,
        ao.object_id
      FROM `{FULL_TRANSFORMED_TABLE_ID}` td
      INNER JOIN `{FULL_ANALYSIS_OBS_TABLE_ID}` ao
        ON td.id = ao.id
      WHERE td.test_orbit_id = @test_orbit_id
        AND ao.object_id IS NOT NULL
    ),
    object_counts AS (
      SELECT object_id, COUNT(*) as n_obs
      FROM labeled_detections
      GROUP BY object_id
      HAVING n_obs >= @min_obs
    )
    SELECT ld.*
    FROM labeled_detections ld
    INNER JOIN object_counts oc
      ON ld.object_id = oc.object_id
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("test_orbit_id", "STRING", test_orbit_id),
            bigquery.ScalarQueryParameter("min_obs", "INT64", MIN_OBS_PER_ORBIT),
        ]
    )

    df_with_labels = bqclient.query(query, job_config=job_config).to_dataframe()
    print(f"DF size after SQL filtering: {df_with_labels.shape}")

    if df_with_labels.empty:
        continue

    # df_with_labels now only contains labeled detections for objects with >= MIN_OBS_PER_ORBIT
    orbits_with_enough_obs = df_with_labels
    print(
        f"DF size after filtering for {MIN_OBS_PER_ORBIT}+ observations: "
        f"{orbits_with_enough_obs.shape}"
    )

    if orbits_with_enough_obs.empty:
        continue

    # One row per (test_orbit, object_id) with linearity metrics
    summary = (
        orbits_with_enough_obs
        .groupby("object_id")
        .apply(compute_linearity)
        .reset_index()
    )

    summary["test_orbit_id"] = test_orbit_id
    summary = summary[["test_orbit_id", "object_id", "n_obs", "linearity_r2"]]
    all_summaries.append(summary)

    # Periodic checkpoint
    if len(all_summaries) % CHECKPOINT_EVERY == 0:
        all_results = pd.concat(all_summaries, ignore_index=True)
        all_results.to_parquet(CHECKPOINT_PATH, index=False)
        print(
            f"Checkpointed {len(all_results)} rows to {CHECKPOINT_PATH} "
            f"after {i} test orbits."
        )

# Final combine
if all_summaries:
    all_results = pd.concat(all_summaries, ignore_index=True)
    all_results.to_parquet(CHECKPOINT_PATH, index=False)
    print(f"Final results written to {CHECKPOINT_PATH}, rows: {len(all_results)}")
    display(all_results.head())
else:
    print("No objects with enough observations found in any test orbit.")



Processing 1 of 5145 (test_orbit_id=40085_r1.349_e0.000_nu105.000_psi-15.000)
DF size after SQL filtering: (39, 6)
DF size after filtering for 6+ observations: (39, 6)


KeyError: 'theta_x'